# Real ML/AI Sketch Enhancement: Pix2Pix GAN Training Notebook

Welcome! This notebook allows you to easily train a custom **Pix2Pix GAN** (U-Net Generator + PatchGAN Discriminator) to convert rough hand-drawn sketches into clean, geometric vector-like sketches. 

We train on a mixture of Google's **QuickDraw dataset** (rendered dynamically) and the **TU-Berlin Sketch dataset** to cover a wide variety of daily object sketches. 

### **Instructions:**
1. Go to **Runtime** > **Change runtime type** > Select **GPU T4**.
2. Run the cells sequentially.
3. At the end, download the exported `sketch_enhancer.onnx` file and place it in the `ml-backend/checkpoints/` directory in your local repository.

In [ ]:
# Run this cell to instantly clear all previous failed downloads and free up all disk space!
!rm -rf data/temp /kaggle/working/data/temp /content/data/temp

## 1. Setup Environment & Import Dependencies

In [ ]:
# Install dependencies
!pip install numpy Pillow matplotlib opencv-python-headless scipy torch torchvision tqdm onnx onnxruntime

In [ ]:
import os
import sys
import torch
import torchvision
from pathlib import Path
import matplotlib.pyplot as plt

# Ensure directories are created
Path("models_arch").mkdir(exist_ok=True)
Path("training").mkdir(exist_ok=True)
Path("deployment").mkdir(exist_ok=True)
Path("checkpoints").mkdir(exist_ok=True)
Path("data").mkdir(exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device} (Should be CUDA for fast GPU training!)")

## 2. Write PyTorch GAN Model Architecture

In [ ]:
%%writefile models_arch/generator.py
import torch
import torch.nn as nn

class UNetBlock(nn.Module):
    def __init__(self, in_channels, out_channels, submodule=None, innermost=False, outermost=False, use_dropout=False):
        super(UNetBlock, self).__init__()
        self.outermost = outermost
        downconv = nn.Conv2d(in_channels, out_channels, kernel_size=4, stride=2, padding=1, bias=False)
        downrelu = nn.LeakyReLU(0.2, True)
        
        if outermost:
            upconv = nn.ConvTranspose2d(out_channels * 2, in_channels, kernel_size=4, stride=2, padding=1)
            down = [downconv]
            up = [nn.ReLU(True), upconv, nn.Tanh()]
            model = down + [submodule] + up
        elif innermost:
            upconv = nn.ConvTranspose2d(out_channels, in_channels, kernel_size=4, stride=2, padding=1, bias=False)
            down = [downrelu, downconv]
            up = [nn.ReLU(True), upconv, nn.BatchNorm2d(in_channels)]
            model = down + up
        else:
            upconv = nn.ConvTranspose2d(out_channels * 2, in_channels, kernel_size=4, stride=2, padding=1, bias=False)
            down = [downrelu, downconv, nn.BatchNorm2d(out_channels)]
            up = [nn.ReLU(True), upconv, nn.BatchNorm2d(in_channels)]
            
            if use_dropout:
                model = down + [submodule] + up + [nn.Dropout(0.5)]
            else:
                model = down + [submodule] + up
                
        self.model = nn.Sequential(*model)

    def forward(self, x):
        if self.outermost:
            return self.model(x)
        else:
            return torch.cat([x, self.model(x)], 1)

class UNetGenerator(nn.Module):
    def __init__(self, in_channels=1, out_channels=1, num_filters=64):
        super(UNetGenerator, self).__init__()
        unet_block = UNetBlock(num_filters * 8, num_filters * 8, submodule=None, innermost=True)
        unet_block = UNetBlock(num_filters * 8, num_filters * 8, submodule=unet_block, use_dropout=True)
        unet_block = UNetBlock(num_filters * 8, num_filters * 8, submodule=unet_block, use_dropout=True)
        unet_block = UNetBlock(num_filters * 8, num_filters * 8, submodule=unet_block, use_dropout=True)
        unet_block = UNetBlock(num_filters * 4, num_filters * 8, submodule=unet_block)
        unet_block = UNetBlock(num_filters * 2, num_filters * 4, submodule=unet_block)
        unet_block = UNetBlock(num_filters, num_filters * 2, submodule=unet_block)
        self.model = UNetBlock(out_channels, num_filters, submodule=unet_block, outermost=True)

    def forward(self, x):
        return self.model(x)

In [ ]:
%%writefile models_arch/discriminator.py
import torch
import torch.nn as nn

class PatchGANDiscriminator(nn.Module):
    def __init__(self, in_channels=2, num_filters=64):
        super(PatchGANDiscriminator, self).__init__()
        self.model = nn.Sequential(
            nn.Conv2d(in_channels, num_filters, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            
            nn.Conv2d(num_filters, num_filters * 2, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(num_filters * 2),
            nn.LeakyReLU(0.2, inplace=True),
            
            nn.Conv2d(num_filters * 2, num_filters * 4, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(num_filters * 4),
            nn.LeakyReLU(0.2, inplace=True),
            
            nn.Conv2d(num_filters * 4, num_filters * 8, kernel_size=4, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(num_filters * 8),
            nn.LeakyReLU(0.2, inplace=True),
            
            nn.Conv2d(num_filters * 8, 1, kernel_size=4, stride=1, padding=1)
        )

    def forward(self, x):
        return self.model(x)

## 3. Write Corruption & Dataset Setup Code

In [ ]:
%%writefile training/corruption.py
import cv2
import numpy as np
from PIL import Image
import random
from scipy.ndimage import map_coordinates, gaussian_filter

class SketchCorruptor:
    def apply_elastic_transform(self, image_np, alpha=30, sigma=4):
        random_state = np.random.RandomState(None)
        shape = image_np.shape
        dx = gaussian_filter((random_state.rand(*shape) * 2 - 1), sigma, mode="constant", cval=0) * alpha
        dy = gaussian_filter((random_state.rand(*shape) * 2 - 1), sigma, mode="constant", cval=0) * alpha
        x, y = np.meshgrid(np.arange(shape[1]), np.arange(shape[0]))
        indices = np.reshape(y + dy, (-1, 1)), np.reshape(x + dx, (-1, 1))
        distorted_img = map_coordinates(image_np, indices, order=1, mode='constant', cval=255)
        return distorted_img.reshape(shape)

    def apply_line_breaks(self, img_np, num_breaks=4):
        h, w = img_np.shape
        corrupted = img_np.copy()
        for _ in range(random.randint(2, num_breaks)):
            x1, y1 = random.randint(0, w), random.randint(0, h)
            x2 = x1 + random.randint(-40, 40)
            y2 = y1 + random.randint(-40, 40)
            thickness = random.randint(4, 10)
            cv2.line(corrupted, (x1, y1), (x2, y2), 255, thickness)
        return corrupted

    def apply_motion_blur(self, img_np, size=5):
        kernel = np.zeros((size, size))
        direction = random.choice(['diag', 'horiz', 'vert'])
        if direction == 'horiz':
            kernel[int((size-1)/2), :] = np.ones(size)
        elif direction == 'vert':
            kernel[:, int((size-1)/2)] = np.ones(size)
        else:
            np.fill_diagonal(kernel, 1)
        kernel = kernel / size
        return cv2.filter2D(img_np, -1, kernel)

    def apply_ink_bleeding(self, img_np):
        inv = 255 - img_np
        blurred = cv2.GaussianBlur(inv, (5, 5), 0)
        thresh = random.randint(50, 90)
        _, binary = cv2.threshold(blurred, thresh, 255, cv2.THRESH_BINARY)
        noise = np.random.normal(0, 15, img_np.shape).astype(np.float32)
        bleed = np.clip(binary.astype(np.float32) + noise, 0, 255).astype(np.uint8)
        return 255 - bleed

    def apply_noise(self, img_np, noise_type="gauss"):
        if noise_type == "gauss":
            row, col = img_np.shape
            mean = 0
            var = random.uniform(50, 200)
            sigma = var ** 0.5
            gauss = np.random.normal(mean, sigma, (row, col))
            noisy = img_np + gauss
            return np.clip(noisy, 0, 255).astype(np.uint8)
        return img_np

    def corrupt(self, img_pil):
        img_np = np.array(img_pil.convert("L"))
        if random.random() < 0.90:
            img_np = self.apply_elastic_transform(img_np, alpha=random.randint(15, 25), sigma=random.randint(3, 4))
        if random.random() < 0.70:
            img_np = self.apply_ink_bleeding(img_np)
        if random.random() < 0.60:
            img_np = self.apply_line_breaks(img_np, num_breaks=random.randint(2, 4))
        if random.random() < 0.50:
            img_np = self.apply_motion_blur(img_np, size=random.choice([3, 5]))
        if random.random() < 0.60:
            img_np = self.apply_noise(img_np, noise_type="gauss")
        h, w = img_np.shape
        cv2.rectangle(img_np, (0, 0), (w-1, h-1), 255, 3)
        return Image.fromarray(img_np).convert("L")

In [ ]:
%%writefile training/dataset.py
import os
from pathlib import Path
from PIL import Image
import torch
from torch.utils.data import Dataset
import torchvision.transforms as transforms
from training.corruption import SketchCorruptor

class SketchPairDataset(Dataset):
    def __init__(self, data_root="data", split="train", val_ratio=0.2, dynamic_corruption=True):
        super(SketchPairDataset, self).__init__()
        self.data_root = Path(data_root)
        self.dynamic_corruption = dynamic_corruption
        self.corruptor = SketchCorruptor() if dynamic_corruption else None
        self.clean_dir = self.data_root / "clean"
        
        if not self.clean_dir.exists():
            raise RuntimeError(f"Directory {self.clean_dir} does not exist!")
            
        self.file_names = sorted([f.name for f in self.clean_dir.glob("*.png")])
        
        total_samples = len(self.file_names)
        val_size = int(total_samples * val_ratio)
        
        import random
        random.seed(42)
        random.shuffle(self.file_names)
        
        if split == "train":
            self.file_names = self.file_names[val_size:]
        elif split == "val":
            self.file_names = self.file_names[:val_size]
            
        self.transform = transforms.Compose([
            transforms.Resize((256, 256)),
            transforms.ToTensor(),
            transforms.Normalize((0.5,), (0.5,))
        ])

    def __len__(self):
        return len(self.file_names)

    def __getitem__(self, idx):
        file_name = self.file_names[idx]
        clean_img = Image.open(self.clean_dir / file_name).convert("L")
        corr_img = self.corruptor.corrupt(clean_img)
        return self.transform(corr_img), self.transform(clean_img)

## 4. Write Dataset Downloader and Generate Images

In [ ]:
%%writefile training/download_datasets.py
import os
import json
import urllib.request
import urllib.parse
from pathlib import Path
from PIL import Image, ImageDraw

CATEGORIES_URL = "https://raw.githubusercontent.com/googlecreativelab/quickdraw-dataset/master/categories.txt"

def get_all_quickdraw_categories():
    print("Fetching official list of all 345 QuickDraw categories from Google...")
    try:
        req = urllib.request.Request(CATEGORIES_URL, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req) as response:
            categories = response.read().decode("utf-8").strip().split("\n")
        categories = [cat.strip() for cat in categories if cat.strip()]
        print(f"[OK] Successfully retrieved all {len(categories)} categories!")
        return categories
    except Exception as e:
        print(f"[WARN] Failed to fetch category list: {e}. Falling back to standard list...")
        return ["apple", "airplane", "banana", "cat", "dog", "fish", "house", "car", "tree", "clock"]

QUICKDRAW_CATEGORIES = get_all_quickdraw_categories()
DATA_DIR = Path("data")
CLEAN_DIR = DATA_DIR / "clean"

def ensure_dirs():
    CLEAN_DIR.mkdir(parents=True, exist_ok=True)
    (DATA_DIR / "temp").mkdir(parents=True, exist_ok=True)

def render_quickdraw_drawing(drawing, size=256, stroke_width=3):
    img = Image.new("L", (size, size), 255)
    draw = ImageDraw.Draw(img)
    all_x = []
    all_y = []
    for stroke in drawing:
        all_x.extend(stroke[0])
        all_y.extend(stroke[1])
    if not all_x or not all_y: return img
    min_x, max_x = min(all_x), max(all_x)
    min_y, max_y = min(all_y), max(all_y)
    w, h = max(1, max_x - min_x), max(1, max_y - min_y)
    margin = 20
    target_inner = size - 2 * margin
    scale = min(target_inner / w, target_inner / h)
    
    for stroke in drawing:
        coords = []
        for x, y in zip(stroke[0], stroke[1]):
            nx = int(margin + (x - min_x) * scale + (target_inner - w * scale) / 2)
            ny = int(margin + (y - min_y) * scale + (target_inner - h * scale) / 2)
            coords.append((nx, ny))
        if len(coords) > 1:
            draw.line(coords, fill=0, width=stroke_width, joint="round")
    return img

def setup_data(samples_per_category=300):
    ensure_dirs()
    print("Downloading and rendering QuickDraw sketches using streaming...")
    for category in QUICKDRAW_CATEGORIES:
        category_safe = category.replace(" ", "_")
        
        # URL encode spaces to %20 for Google Storage bucket compatibility
        url_encoded_category = urllib.parse.quote(category)
        url = f"https://storage.googleapis.com/quickdraw_dataset/full/simplified/{url_encoded_category}.ndjson"
        
        print(f"Streaming and rendering category: {category}...")
        count = 0
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req) as response:
                for line_bytes in response:
                    if count >= samples_per_category:
                        break
                    
                    line_str = line_bytes.decode("utf-8")
                    data = json.loads(line_str)
                    
                    if "drawing" in data:
                        pil = render_quickdraw_drawing(data["drawing"])
                        pil.save(CLEAN_DIR / f"qd_{category_safe}_{count}.png")
                        count += 1
            print(f"Generated {count} images for {category_safe}")
        except Exception as e:
            print(f"Failed to process category {category_safe}: {e}")

if __name__ == "__main__":
    setup_data(samples_per_category=300)

In [ ]:
# Run the download and dataset setup
!python training/download_datasets.py

## 5. View Sample Training Pairs

In [ ]:
from training.dataset import SketchPairDataset
import matplotlib.pyplot as plt

dataset = SketchPairDataset(split="train", dynamic_corruption=True)
corr, clean = dataset[0]

# Denormalize for display: [-1, 1] -> [0, 1]
corr_show = (corr.squeeze(0).numpy() + 1.0) / 2.0
clean_show = (clean.squeeze(0).numpy() + 1.0) / 2.0

fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].imshow(corr_show, cmap="gray")
ax[0].set_title("Corrupted Sketch Input (Human Hand Mock)")
ax[0].axis("off")

ax[1].imshow(clean_show, cmap="gray")
ax[1].set_title("Target Clean Geometric Drawing")
ax[1].axis("off")

plt.show()

## 6. Run Pix2Pix GAN Training

In [ ]:
%%writefile training/train.py
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision.utils import save_image
from models_arch.generator import UNetGenerator
from models_arch.discriminator import PatchGANDiscriminator
from training.dataset import SketchPairDataset

def train():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Training on {device}")
    
    os.makedirs("checkpoints", exist_ok=True)
    
    train_dataset = SketchPairDataset(split="train")
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, drop_last=True)
    
    net_g = UNetGenerator(1, 1).to(device)
    net_d = PatchGANDiscriminator(2).to(device)
    
    criterion_gan = nn.MSELoss()
    criterion_l1 = nn.L1Loss()
    
    opt_g = optim.Adam(net_g.parameters(), lr=0.0002, betas=(0.5, 0.999))
    opt_d = optim.Adam(net_d.parameters(), lr=0.0002, betas=(0.5, 0.999))
    
    for epoch in range(1, 41):  # 40 Epochs is fast and plenty on T4 GPU!
        net_g.train()
        for idx, (corr, clean) in enumerate(train_loader):
            corr, clean = corr.to(device), clean.to(device)
            
            # Train D
            opt_d.zero_grad()
            pred_real = net_d(torch.cat((corr, clean), 1))
            loss_d_real = criterion_gan(pred_real, torch.ones_like(pred_real))
            
            fake_clean = net_g(corr)
            pred_fake = net_d(torch.cat((corr, fake_clean.detach()), 1))
            loss_d_fake = criterion_gan(pred_fake, torch.zeros_like(pred_fake))
            
            loss_d = (loss_d_real + loss_d_fake) * 0.5
            loss_d.backward()
            opt_d.step()
            
            # Train G
            opt_g.zero_grad()
            pred_g = net_d(torch.cat((corr, fake_clean), 1))
            loss_g_gan = criterion_gan(pred_g, torch.ones_like(pred_g))
            loss_g_l1 = criterion_l1(fake_clean, clean)
            loss_g = loss_g_gan + 100.0 * loss_g_l1
            loss_g.backward()
            opt_g.step()
            
        print(f"Epoch [{epoch}/40] D_Loss: {loss_d.item():.4f} G_Loss: {loss_g.item():.4f} L1_Loss: {loss_g_l1.item():.4f}")
        
        if epoch % 10 == 0:
            torch.save(net_g.state_dict(), f"checkpoints/generator_epoch_{epoch}.pth")
            
    torch.save(net_g.state_dict(), "checkpoints/sketch_enhancer_best.pth")
    print("Training Complete!")

if __name__ == "__main__":
    train()

In [ ]:
# Run Pix2Pix GAN training!
!python training/train.py

## 7. Export the trained generator weights to ONNX format

In [ ]:
%%writefile deployment/export_onnx.py
import sys
from pathlib import Path
# Add parent directory to system path to avoid ModuleNotFoundError inside notebooks
sys.path.append(str(Path(__file__).resolve().parent.parent))

import torch
from models_arch.generator import UNetGenerator
import argparse

def export():
    weight_path = Path("checkpoints/sketch_enhancer_best.pth")
    onnx_path = Path("checkpoints/sketch_enhancer.onnx")
    
    print(f"Loading weights from {weight_path}...")
    model = UNetGenerator(in_channels=1, out_channels=1)
    model.load_state_dict(torch.load(weight_path, map_location="cpu"))
    model.eval()
    
    # Dummy input of shape (B=1, C=1, H=256, W=256)
    dummy_input = torch.randn(1, 1, 256, 256)
    
    print(f"Exporting to ONNX format at {onnx_path}...")
    torch.onnx.export(
        model,
        dummy_input,
        str(onnx_path),
        export_params=True,
        opset_version=14,
        do_constant_folding=True,
        input_names=["input"],
        output_names=["output"],
        dynamic_axes={
            "input": {0: "batch_size"},
            "output": {0: "batch_size"}
        }
    )
    print(f"[SUCCESS] Exported ONNX model size: {onnx_path.stat().st_size / (1024*1024):.2f} MB")

if __name__ == "__main__":
    export()

In [ ]:
# Export PyTorch model to ONNX
!python deployment/export_onnx.py

## 8. Download the finished ONNX model file to your PC!

In [ ]:
from google.colab import files
files.download('checkpoints/sketch_enhancer.onnx')